In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import importlib

# Locate project root
project_root = Path.cwd().parents[2]

# Add project root to Python's module search path
sys.path.insert(0, str(project_root))

# Import reusable modules
from src.data_loader import load_cv_data
import src.feature_engineering as fe

# Load the latest feature-engineering implementation
importlib.reload(fe)

print("Project root:", project_root)
print("Setup completed.")

Project root: c:\Users\rajur\OneDrive\Desktop\next-move-ai
Setup completed.


In [2]:
fold_datasets = load_cv_data(project_root)

# Convert skill IDs into Python sets
def convert_skill_id(skill_id):
    if pd.isna(skill_id):
        return set()

    return set(str(skill_id).split("_"))


for train_df, val_df in fold_datasets:

    train_df["skill_set"] = train_df["skill_id"].apply(
        convert_skill_id
    )

    val_df["skill_set"] = val_df["skill_id"].apply(
        convert_skill_id
    )


# Display fold sizes
for fold_number, (train_df, val_df) in enumerate(
    fold_datasets, start=1
):
    print(
        f"Fold {fold_number}: "
        f"Training = {train_df.shape}, "
        f"Validation = {val_df.shape}"
    )

Fold 1: Training = (177324, 7), Validation = (42659, 7)
Fold 2: Training = (176273, 7), Validation = (43710, 7)
Fold 3: Training = (170186, 7), Validation = (49797, 7)
Fold 4: Training = (178311, 7), Validation = (41672, 7)
Fold 5: Training = (177838, 7), Validation = (42145, 7)


In [3]:
v24_fold_datasets = []

for fold_number, (fold_train_df, fold_val_df) in enumerate(
    fold_datasets, start=1
):

    print(f"\nProcessing Fold {fold_number}...")

    # Generate fractional same-skill features for training data
    v24_train_df = fe.add_fractional_same_skill_history_features(
        fold_train_df,
        alpha=10,
        prior_data=fold_train_df
    )

    # Generate fractional same-skill features for validation data
    v24_val_df = fe.add_fractional_same_skill_history_features(
        fold_val_df,
        alpha=10,
        prior_data=fold_train_df
    )

    v24_fold_datasets.append(
        (v24_train_df, v24_val_df)
    )

    print("Training features:", v24_train_df.shape)
    print("Validation features:", v24_val_df.shape)

print("\nV2.4 fractional feature generation completed.")


Processing Fold 1...
Training features: (177324, 11)
Validation features: (42659, 11)

Processing Fold 2...
Training features: (176273, 11)
Validation features: (43710, 11)

Processing Fold 3...
Training features: (170186, 11)
Validation features: (49797, 11)

Processing Fold 4...
Training features: (178311, 11)
Validation features: (41672, 11)

Processing Fold 5...
Training features: (177838, 11)
Validation features: (42145, 11)

V2.4 fractional feature generation completed.


In [4]:
feature_columns_to_check = [
    "weighted_same_skill_previous_interactions",
    "weighted_same_skill_previous_correct",
    "weighted_same_skill_global_prior",
    "weighted_same_skill_smoothed_accuracy"
]

for fold_number, (train_df, val_df) in enumerate(
    v24_fold_datasets, start=1
):

    print(f"\n========== FOLD {fold_number} ==========")

    for name, df in [
        ("Training", train_df),
        ("Validation", val_df)
    ]:

        print(f"\n{name}:")

        print(
            "Missing values:",
            df[feature_columns_to_check].isna().sum().sum()
        )

        print(
            "Invalid historical counts:",
            (
                df["weighted_same_skill_previous_correct"]
                > df["weighted_same_skill_previous_interactions"]
            ).sum()
        )

        print(
            "Global prior range:",
            df["weighted_same_skill_global_prior"].min(),
            df["weighted_same_skill_global_prior"].max()
        )

        print(
            "Smoothed accuracy range:",
            df["weighted_same_skill_smoothed_accuracy"].min(),
            df["weighted_same_skill_smoothed_accuracy"].max()
        )


========== FOLD 1 ==========

Training:
Missing values: 0
Invalid historical counts: 0
Global prior range: 0.0 1.0
Smoothed accuracy range: 0.0 1.0

Validation:
Missing values: 0
Invalid historical counts: 0
Global prior range: 0.0 1.0
Smoothed accuracy range: 0.0 1.0

========== FOLD 2 ==========

Training:
Missing values: 0
Invalid historical counts: 0
Global prior range: 0.0 1.0
Smoothed accuracy range: 0.0 1.0

Validation:
Missing values: 0
Invalid historical counts: 0
Global prior range: 0.0 1.0
Smoothed accuracy range: 0.0 1.0

========== FOLD 3 ==========

Training:
Missing values: 0
Invalid historical counts: 0
Global prior range: 0.0 1.0
Smoothed accuracy range: 0.0 1.0

Validation:
Missing values: 0
Invalid historical counts: 0
Global prior range: 0.0 1.0
Smoothed accuracy range: 0.0 1.0

========== FOLD 4 ==========

Training:
Missing values: 0
Invalid historical counts: 0
Global prior range: 0.0 1.0
Smoothed accuracy range: 0.0 1.0

Validation:
Missing values: 0
Invalid hi

In [5]:
for train_df, val_df in v24_fold_datasets:

    train_df["log_weighted_same_skill_previous_interactions"] = (
        np.log1p(
            train_df["weighted_same_skill_previous_interactions"]
        )
    )

    val_df["log_weighted_same_skill_previous_interactions"] = (
        np.log1p(
            val_df["weighted_same_skill_previous_interactions"]
        )
    )

print("Log fractional experience feature created for all folds.")

Log fractional experience feature created for all folds.


In [6]:
for fold_number, (train_df, val_df) in enumerate(
    v24_fold_datasets, start=1
):

    print(f"\n========== FOLD {fold_number} ==========")

    for name, df in [
        ("Training", train_df),
        ("Validation", val_df)
    ]:

        print(f"\n{name}:")

        print(
            "Missing values:",
            df["log_weighted_same_skill_previous_interactions"].isna().sum()
        )

        print(
            "Minimum log experience:",
            df["log_weighted_same_skill_previous_interactions"].min()
        )

        print(
            "Maximum log experience:",
            df["log_weighted_same_skill_previous_interactions"].max()
        )

        print(
            "Negative values:",
            (
                df["log_weighted_same_skill_previous_interactions"] < 0
            ).sum()
        )


========== FOLD 1 ==========

Training:
Missing values: 0
Minimum log experience: 0.0
Maximum log experience: 5.081404364984463
Negative values: 0

Validation:
Missing values: 0
Minimum log experience: 0.0
Maximum log experience: 4.859812404361672
Negative values: 0

========== FOLD 2 ==========

Training:
Missing values: 0
Minimum log experience: 0.0
Maximum log experience: 4.990432586778736
Negative values: 0

Validation:
Missing values: 0
Minimum log experience: 0.0
Maximum log experience: 5.081404364984463
Negative values: 0

========== FOLD 3 ==========

Training:
Missing values: 0
Minimum log experience: 0.0
Maximum log experience: 5.081404364984463
Negative values: 0

Validation:
Missing values: 0
Minimum log experience: 0.0
Maximum log experience: 4.969813299576001
Negative values: 0

========== FOLD 4 ==========

Training:
Missing values: 0
Minimum log experience: 0.0
Maximum log experience: 5.081404364984463
Negative values: 0

Validation:
Missing values: 0
Minimum log exper

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    log_loss,
    brier_score_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Two input features for V2.4
feature_columns = [
    "weighted_same_skill_smoothed_accuracy",
    "log_weighted_same_skill_previous_interactions"
]

v24_models = []
v24_results = []

for fold_number, (train_data, val_data) in enumerate(
    v24_fold_datasets, start=1
):

    X_train = train_data[feature_columns]
    y_train = train_data["correct"]

    X_val = val_data[feature_columns]
    y_val = val_data["correct"]

    # Same preprocessing and classifier as previous experiments
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000))
    ])

    # Train using training data only
    model.fit(X_train, y_train)

    # Predict probabilities and class labels
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    y_pred = model.predict(X_val)

    # Probability-based evaluation
    ll = log_loss(y_val, y_pred_proba, labels=[0, 1])
    bs = brier_score_loss(y_val, y_pred_proba)

    # Classification-based evaluation
    accuracy = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)
    macro_f1 = f1_score(
        y_val,
        y_pred,
        average="macro",
        zero_division=0
    )

    v24_models.append(model)

    v24_results.append({
        "fold": fold_number,
        "log_loss": ll,
        "brier_score": bs,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "macro_f1": macro_f1
    })

    print(f"\nFold {fold_number}")
    print(f"Log Loss:  {ll:.4f}")
    print(f"Brier:     {bs:.4f}")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(f"Macro F1:  {macro_f1:.4f}")


Fold 1
Log Loss:  0.5911
Brier:     0.2024
Accuracy:  0.6964
Precision: 0.7027
Recall:    0.9265
F1-score:  0.7992
Macro F1:  0.5883

Fold 2
Log Loss:  0.5852
Brier:     0.1998
Accuracy:  0.7048
Precision: 0.7185
Recall:    0.9255
F1-score:  0.8090
Macro F1:  0.5801

Fold 3
Log Loss:  0.5979
Brier:     0.2055
Accuracy:  0.6850
Precision: 0.6902
Recall:    0.9239
F1-score:  0.7902
Macro F1:  0.5796

Fold 4
Log Loss:  0.5904
Brier:     0.2020
Accuracy:  0.6983
Precision: 0.7066
Recall:    0.9320
F1-score:  0.8038
Macro F1:  0.5755

Fold 5
Log Loss:  0.5858
Brier:     0.2002
Accuracy:  0.7012
Precision: 0.7098
Recall:    0.9293
F1-score:  0.8049
Macro F1:  0.5836


In [8]:
v24_results_df = pd.DataFrame(v24_results)

print("V2.4 Cross-Validation Results")
display(v24_results_df.round(4))

print("\nAverage Performance:")

metrics = [
    "log_loss",
    "brier_score",
    "accuracy",
    "precision",
    "recall",
    "f1_score",
    "macro_f1"
]

for metric in metrics:
    print(
        f"Average {metric}: "
        f"{v24_results_df[metric].mean():.4f}"
    )

V2.4 Cross-Validation Results


,fold,log_loss,brier_score,accuracy,precision,recall,f1_score,macro_f1
0,1,0.5911,0.2024,0.6964,0.7027,0.9265,0.7992,0.5883
1,2,0.5852,0.1998,0.7048,0.7185,0.9255,0.8090,0.5801
2,3,0.5979,0.2055,0.6850,0.6902,0.9239,0.7902,0.5796
3,4,0.5904,0.2020,0.6983,0.7066,0.9320,0.8038,0.5755
4,5,0.5858,0.2002,0.7012,0.7098,0.9293,0.8049,0.5836



Average Performance:
Average log_loss: 0.5901
Average brier_score: 0.2020
Average accuracy: 0.6971
Average precision: 0.7056
Average recall: 0.9274
Average f1_score: 0.8014
Average macro_f1: 0.5814


In [9]:
# Collect out-of-fold predictions for V2.4

v24_predictions = []

for fold_number, (train_data, val_data) in enumerate(
    v24_fold_datasets, start=1
):

    # Retrieve the already-trained model for this fold
    model = v24_models[fold_number - 1]

    # Generate validation probabilities
    X_val = val_data[feature_columns]
    y_pred_proba = model.predict_proba(X_val)[:, 1]

    # Store actual outcomes and predictions
    fold_predictions = pd.DataFrame({
        "user_id": val_data["user_id"].values,
        "order_id": val_data["order_id"].values,
        "fold": fold_number,
        "y_true": val_data["correct"].values,
        "y_pred_proba": y_pred_proba,
        "model": "V2.4"
    })

    v24_predictions.append(fold_predictions)

# Combine predictions from all five folds
v24_predictions_df = pd.concat(
    v24_predictions,
    ignore_index=True
)

print("Prediction collection completed.")
print("Total predictions:", len(v24_predictions_df))
display(v24_predictions_df.head())

Prediction collection completed.
Total predictions: 219983


,user_id,order_id,fold,y_true,y_pred_proba,model
0,21825,21616385,1,0,0.433472,V2.4
1,21825,21616478,1,1,0.395785,V2.4
2,21825,21616677,1,1,0.458696,V2.4
3,21825,21616919,1,1,0.512424,V2.4
4,21825,21671620,1,0,0.522964,V2.4


In [10]:
print("Total rows:", len(v24_predictions_df))

print("\nMissing values:")
print(v24_predictions_df.isna().sum())

print("\nFold distribution:")
print(v24_predictions_df["fold"].value_counts().sort_index())

print("\nProbability range:")
print(
    v24_predictions_df["y_pred_proba"].min(),
    v24_predictions_df["y_pred_proba"].max()
)

print("\nActual outcome distribution:")
print(v24_predictions_df["y_true"].value_counts())

Total rows: 219983

Missing values:
user_id         0
order_id        0
fold            0
y_true          0
y_pred_proba    0
model           0
dtype: int64

Fold distribution:
fold
1    42659
2    43710
3    49797
4    41672
5    42145
Name: count, dtype: int64

Probability range:
0.08365422768376468 0.9252924016555887

Actual outcome distribution:
y_true
1    144886
0     75097
Name: count, dtype: int64


In [11]:
# Create directory for collected model predictions
prediction_dir = (
    project_root / "data" / "processed" / "model_predictions"
)

prediction_dir.mkdir(parents=True, exist_ok=True)

# Save V2.4 predictions
v24_prediction_path = prediction_dir / "v2_4_predictions.csv"

v24_predictions_df.to_csv(
    v24_prediction_path,
    index=False
)

print("V2.4 predictions saved successfully.")
print("File:", v24_prediction_path)
print("Total records:", len(v24_predictions_df))

V2.4 predictions saved successfully.
File: c:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\model_predictions\v2_4_predictions.csv
Total records: 219983


In [12]:
feature_dir = (
    project_root / "data" / "processed" / "engineered_features"
    / "group_2_same_skill_history" / "v2.4"
)
feature_dir.mkdir(parents=True, exist_ok=True)

columns_to_save = [
    "user_id",
    "order_id",
    "correct",
    "weighted_same_skill_smoothed_accuracy",
    "log_weighted_same_skill_previous_interactions"
]

for fold_number, (train_df, val_df) in enumerate(v24_fold_datasets, start=1):
    train_df[columns_to_save].to_csv(
        feature_dir / f"fold_{fold_number}_train.csv", index=False
    )
    val_df[columns_to_save].to_csv(
        feature_dir / f"fold_{fold_number}_validation.csv", index=False
    )
    print(f"Fold {fold_number} saved.")

print("V2.4 features saved successfully!")

Fold 1 saved.
Fold 2 saved.
Fold 3 saved.
Fold 4 saved.
Fold 5 saved.
V2.4 features saved successfully!
